In [2]:
# 环境 安装
# python -m pip install modelscope huggingface_hub transformers torch -i https://pypi.tuna.tsinghua.edu.cn/simple


# 用了 device_map="cuda" 这个参数，它需要 accelerate 这个库的支持
# pip install accelerate  

# DeepSeek-R1-Distill-Qwen-7B 是一个70亿参数的模型。在默认的 FP16（半精度）格式下，它大约需要 14GB 的显存。而你的 GTX 2060 只有 6GB 显存，差距非常大。
# 方案一：使用 4-bit 量化加载（最推荐）
# 这是最直接有效的方法，可以将显存占用降低到 6GB 左右。你需要安装 bitsandbytes 库：
# pip install bitsandbytes



SyntaxError: invalid syntax (1952060261.py, line 1)

In [2]:
from modelscope import snapshot_download

# ========== 配置 ==========
# 下载到哪个目录
DOWNLOAD_DIR = r"./model_save"

# 要下载的模型名（从 ModelScope 下载，国内速度快）
# 可选模型：
#   - "Qwen/Qwen2.5-7B-Instruct"         # 通义千问 7B
#   - "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  # DeepSeek R1 蒸馏 7B
#   - "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B" # DeepSeek R1 蒸馏 1.5B（小模型测试用）
#   - "ZhipuAI/glm-4-9b-chat"            # 智谱 GLM-4 9B
# MODEL_NAME = "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B"  不可行，模型太大
MODEL_NAME = "unsloth/gemma-4-E4B-it-GGUF"

import os
from modelscope import snapshot_download

model_dir = os.path.join(DOWNLOAD_DIR, MODEL_NAME)
os.makedirs(DOWNLOAD_DIR, exist_ok=True)


In [3]:
import os
import sys
import time
# 创建进度回调类（需要传入类本身，snapshot_download 内部会实例化）
class ProgressCallback:
        def __init__(self):
            self.last_time = time.time()
            self.last_percent = 0

        def __call__(self, current, total, message=None):
            """modelscope 的 snapshot_download 回调 signature:
            __call__(current, total, message)"""
            if total > 0:
                percent = int(current / total * 100)
                now = time.time()
                # 每 5% 或每 3 秒打印一次进度，避免刷屏
                if percent - self.last_percent >= 5 or now - self.last_time > 3:
                    bar_length = 40
                    filled = int(bar_length * current // total)
                    bar = "█" * filled + "░" * (bar_length - filled)
                    speed = current / (now - self.last_time) / 1024 / 1024 if (now - self.last_time) > 0 else 0
                    sys.stdout.write(
                        f"\r  ⬇️  下载进度: |{bar}| {percent}%  "
                        f"({current / 1024 / 1024:.1f}/{total / 1024 / 1024:.1f} MB)  "
                        f"{speed:.1f} MB/s  {message or ''}"
                    )
                    sys.stdout.flush()
                    self.last_time = now
                    self.last_percent = percent

In [5]:
import os
from modelscope import snapshot_download

print(f"开始从 ModelScope 下载模型: {MODEL_NAME}")
print(f"下载到: {model_dir}")
print("首次下载较大（约15GB），请耐心等待...")
print()

local_path = snapshot_download(
    model_id=MODEL_NAME,
    cache_dir=DOWNLOAD_DIR,
    revision="master",
    # 传入回调类列表，snapshot_download 内部会实例化并调用
    # progress_callbacks=[ProgressCallback],
)

print(f"\n✅ 下载完成！模型路径: {local_path}")

开始从 ModelScope 下载模型: unsloth/gemma-4-E4B-it-GGUF
下载到: ./model_save\unsloth/gemma-4-E4B-it-GGUF
首次下载较大（约15GB），请耐心等待...



2026-07-08 20:33:14,896 - modelscope - INFO - Got 18 files, start to download ...


Processing 18 items:   0%|          | 0.00/18.0 [00:00<?, ?it/s]

2026-07-08 21:48:15,543 - modelscope - INFO - Finish downloading 18 files for repo 'unsloth/gemma-4-E4B-it-GGUF'



✅ 下载完成！模型路径: ./model_save\unsloth\gemma-4-E4B-it-GGUF
